## 0  Imports & paths

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, '.')
from utils import DATA_DIR

PV_XLSX  = DATA_DIR / 'democratic-republic-of-congo_hrp_political_violence_events_and_fatalities_by_month-year_as-of-0.xlsx'
CT_XLSX  = DATA_DIR / 'democratic-republic-of-congo_hrp_civilian_targeting_events_and_fatalities_by_month-year_as-of-0.xlsx'
OUT_CSV  = DATA_DIR / 'conflict_dat_cleaned.csv'

print(f'pandas  {pd.__version__}')

pandas  2.3.3


## 1  Load ACLED data

In [2]:
pv = pd.read_excel(PV_XLSX, sheet_name='Data')
ct = pd.read_excel(CT_XLSX, sheet_name='Data')

print(f'Political violence : {pv.shape[0]:,} rows x {pv.shape[1]} cols')
print(f'Civilian targeting : {ct.shape[0]:,} rows x {ct.shape[1]} cols')
print(f'Year range PV: {pv["Year"].min()} – {pv["Year"].max()}')
print(f'Year range CT: {ct["Year"].min()} – {ct["Year"].max()}')
pv.head(3)

Political violence : 66,364 rows x 10 cols
Civilian targeting : 66,364 rows x 10 cols
Year range PV: 1997 – 2026
Year range CT: 1997 – 2026


,Country,Admin1,Admin2,ISO3,Admin2 Pcode,Admin1 Pcode,Month,Year,Events,Fatalities
0,Democratic Republic of Congo,Kasai,Dekese,COD,CD9208,CD92,January,1997,0,0
1,Democratic Republic of Congo,Mai-Ndombe,Oshwe,COD,CD3304,CD33,January,1997,0,0
2,Democratic Republic of Congo,Mongala,Lisala,COD,CD4402,CD44,January,1997,0,0


## 2  Merge PV + CT and filter to 2021–2026

In [3]:
# Both files have identical Admin2 x Month x Year keys — verified by shape equality
# Sum Events and Fatalities across the two violence categories
merged = pv[['Country','Admin1','Admin2','Admin2 Pcode','Admin1 Pcode','Month','Year']].copy()
merged['events']     = pv['Events']     + ct['Events']
merged['fatalities'] = pv['Fatalities'] + ct['Fatalities']

merged = merged[merged['Year'].between(2021, 2026)].reset_index(drop=True)
print(f'Rows after 2021–2026 filter: {len(merged):,}')
merged.head(3)

Rows after 2021–2026 filter: 12,220


,Country,Admin1,Admin2,Admin2 Pcode,Admin1 Pcode,Month,Year,events,fatalities
0,Democratic Republic of Congo,Kasai,Dekese,CD9208,CD92,January,2021,1,9
1,Democratic Republic of Congo,Mai-Ndombe,Oshwe,CD3304,CD33,January,2021,0,0
2,Democratic Republic of Congo,Mongala,Lisala,CD4402,CD44,January,2021,0,0


## 3  Map ACLED Admin-2 names to COD shapefile territory names

In [4]:
# 25 ACLED Admin-2 units are sub-territory cities not present in the COD shapefile.
# These represent 5.2% of fatalities. Mapped to their parent territory below.
CITY_TO_TERRITORY = {
    'Dingila'     : 'Ango',
    'Aba'         : 'Faradje',
    'Isiro'       : 'Rungu',
    'Ariwara'     : 'Aru',
    'Mongwalu'    : 'Djugu',
    'Ingbokolo'   : 'Mahagi',
    'Bunia'       : 'Djugu',
    'Tshikapa'    : 'Kamonia',
    'Tshimbulu'   : 'Dimbelenge',
    'Lukalaba'    : 'Miabi',
    'Bangu'       : 'Lukula',
    'Inkisi'      : 'Songololo',
    'Dibaya-Lubwe': 'Bagata',
    'Mangai'      : 'Bagata',
    'Kolwezi'     : 'Mutshatsha',
    'Kasaji'      : 'Dilolo',
    'Nioki'       : 'Kiri',
    'Kalima'      : 'Pangi',
    'Namoya'      : 'Kabambare',
    'Bena-Dibele' : 'Lubefu',
    'Tshumbe'     : 'Lodja',
    'Kamituga'    : 'Mwenga',
    'Baraka'      : 'Fizi',
    'Kaoze'       : 'Kongolo',
    'Yangambi'    : 'Isangi',
}

merged['town_admin2'] = merged['Admin2'].replace(CITY_TO_TERRITORY)

n_remapped = (merged['Admin2'] != merged['town_admin2']).sum()
print(f'Rows remapped to parent territory: {n_remapped:,}')
print(f'Unique town_admin2 values: {merged["town_admin2"].nunique()}')

Rows remapped to parent territory: 1,625
Unique town_admin2 values: 163


## 4  Build date columns

In [5]:
MONTH_MAP = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

merged['month_num'] = merged['Month'].map(MONTH_MAP)
merged['date_start'] = pd.to_datetime(
    dict(year=merged['Year'], month=merged['month_num'], day=1)
)
merged['year_month'] = merged['date_start'].dt.to_period('M').astype(str)

print(f'Date range: {merged["date_start"].min().date()} to {merged["date_start"].max().date()}')
merged[['Admin2','town_admin2','Year','Month','events','fatalities','date_start','year_month']].head(5)

Date range: 2021-01-01 to 2026-05-01


,Admin2,town_admin2,Year,Month,events,fatalities,date_start,year_month
0,Dekese,Dekese,2021,January,1,9,2021-01-01,2021-01
1,Oshwe,Oshwe,2021,January,0,0,2021-01-01,2021-01
2,Lisala,Lisala,2021,January,0,0,2021-01-01,2021-01
3,Mutshatsha,Mutshatsha,2021,January,1,0,2021-01-01,2021-01
4,Bolobo,Bolobo,2021,January,0,0,2021-01-01,2021-01


## 5  Aggregate to territory × month

In [6]:
# Sum events and fatalities to town_admin2 x year_month
# (some cities were remapped to same territory, so re-aggregate)
result = (
    merged
    .groupby(['town_admin2', 'year_month', 'date_start'], as_index=False)
    .agg(violent_incidents=('events', 'sum'), total_deaths=('fatalities', 'sum'))
)

print(f'Output rows: {len(result):,}')
print(f'Unique territories: {result["town_admin2"].nunique()}')
print(f'Date range: {result["date_start"].min().date()} to {result["date_start"].max().date()}')
print()
print('Top 10 territories by total fatalities:')
print(result.groupby("town_admin2")["total_deaths"].sum().sort_values(ascending=False).head(10))

Output rows: 10,595
Unique territories: 163
Date range: 2021-01-01 to 2026-05-01

Top 10 territories by total fatalities:
town_admin2
Beni        10864
Djugu        8714
Irumu        6454
Rutshuru     5148
Lubero       3637
Mambasa      2868
Masisi       2484
Goma         1902
Uvira        1496
Fizi         1424
Name: total_deaths, dtype: int64


## 6  Export

In [7]:
result.to_csv(OUT_CSV, index=False)
print(f'Saved {len(result):,} rows → {OUT_CSV}')
print(f'Columns: {result.columns.tolist()}')

Saved 10,595 rows → /Users/jackzipper/QSS20/final_project/final_project_data/conflict_dat_cleaned.csv
Columns: ['town_admin2', 'year_month', 'date_start', 'violent_incidents', 'total_deaths']
